# Is the graft off-distribution? Cosine is only half the story

**The question.** The Llama cross-family task map has `recon_cos_taskmap = 0.808` — it sits ON the
recipient's natural manifold *directionally* — yet confers 0.886. Every other pair is 0.197–0.266.
That looks like a counterexample to "conferral coincides with leaving the manifold". It is NOT seed
luck (5 seeds spread 0.006) and NOT layer luck (L17 0.824, L23 0.808).

**What we never measured: NORM.** In the Gemma pair the written vector has
`||f(h)|| = 1686` against a native `||x|| = 291` — **5.8x oversized**. A vector can be nearly
parallel to the manifold and still be far outside the distribution because it is far too long.
Cosine cannot see that.

**Three tests this notebook adds:**
1. **Norm ratio** `||f(h)|| / ||native||` per arm — the missing half.
2. **Standardised z-distance** — per-dimension distance from the native mean in native SD units,
   divided by sqrt(d). Native states score ~1.0 by construction; this captures rotation AND scale
   in one number, so it is the metric the paper should actually report.
3. **Training trajectory + extended training** — cosine/norm logged every epoch, then extra epochs.
   If Llama's cosine is still falling at epoch 6, the map was simply undertrained and the anomaly
   dissolves. If it is flat, 0.808 is a real optimum.

**Three arms, one fixed recipient (`gemma-2-2b`, L_R=20):** gemma-2-9b L34 (within-family
reference, known cos 0.197), Qwen2.5-7B L23 (cross-family, cos 0.253), Llama-3.1-8B L23
(cross-family, cos 0.808 — the anomaly). Qwen is the control that separates "cross-family" from
"Llama-specific".

**48 GB. ~80–100 min.** Recipient loads once; donors load and free one at a time (peak ~24 GB).
2 seeds per arm — the geometric quantity has a 0.006 spread across 5 seeds, so 2 is ample.
Run CELL 1 -> **restart kernel** -> run down.

In [1]:
# === CELL 1: install (run once, then RESTART KERNEL) ===
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer
!pip uninstall -y torchvision torchaudio
# torchvision removed before any import (version mismatch crashes transformers). RESTART after this.
# >>> RESTART THE KERNEL NOW, then run every cell below in order. <<<
# --- Blackwell/sm_120 pods ONLY (verify cell prints cap (12,0)): uncomment, run, restart again ---
# !pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
import torch, transformers
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
print("transformers:", transformers.__version__, "(want 4.46.3)")

torch: 2.4.1+cu124 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)


In [2]:
# === CELL 2: imports, shim, config — THREE ARMS against a fixed recipient ===
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        for a in target.split(".")[:-1]: mod = getattr(mod, a)
        setattr(mod, target.split(".")[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"; torch.manual_seed(0)

MODEL_R = "google/gemma-2-2b"      # recipient, fixed for every arm
L_R     = 20

# (name, donor id, donor layer, published task-map cosine for cross-checking)
ARMS = [
    ("gemma9b", "google/gemma-2-9b",       34, 0.197),   # within-family reference
    ("qwen7b",  "Qwen/Qwen2.5-7B",         23, 0.253),   # cross-family control
    ("llama8b", "meta-llama/Llama-3.1-8B", 23, 0.808),   # THE ANOMALY
]
RUN_ARMS = ["gemma9b", "qwen7b", "llama8b"]

GEOM_SEEDS      = [0, 1]     # 2 seeds: the cosine's 5-seed spread is 0.006, so 2 is ample
TASK_EPOCHS     = 6          # matches every previous run, so cosines stay comparable
EXTENDED_EPOCHS = 6          # extra epochs AFTER the standard 6, to test undertraining
EXTEND_ARMS     = ["llama8b", "gemma9b"]   # the anomaly + a converged control

PATCH_POS = -1
RIDGE_LAMBDA = 1e3
BOOT_B = 10000            # bootstrap resamples, matches every previous run
MAX_NEW_ARITH = 8         # generation budget for a 3-digit answer, matches previous runs
N_ARITH_TRAIN, N_ARITH_EVAL = 3000, 2000
ARITH_BATCH = 16
SMOKE_TEST = False
if SMOKE_TEST:
    N_ARITH_TRAIN, N_ARITH_EVAL = 300, 200
    GEOM_SEEDS, TASK_EPOCHS, EXTENDED_EPOCHS = [0], 2, 2

RESULTS = {"_config": {"recipient": MODEL_R, "L_R": L_R,
                       "arms": {n: {"donor": m, "L_D": l, "published_taskmap_cos": c}
                                for n, m, l, c in ARMS},
                       "geom_seeds": GEOM_SEEDS, "task_epochs": TASK_EPOCHS,
                       "extended_epochs": EXTENDED_EPOCHS, "extend_arms": EXTEND_ARMS,
                       "smoke_test": SMOKE_TEST}}
print("arms:", RUN_ARMS, "| seeds:", GEOM_SEEDS, "| epochs:", TASK_EPOCHS,
      "(+%d extended on %s)" % (EXTENDED_EPOCHS, EXTEND_ARMS))

arms: ['gemma9b', 'qwen7b', 'llama8b'] | seeds: [0, 1] | epochs: 6 (+6 extended on ['llama8b', 'gemma9b'])


In [ ]:
# === CELL 3: Hugging Face login (Gemma and Llama are gated) ===
from huggingface_hub import login
login("")   # <-- paste your own read token here before running

In [5]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [6]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, full-answer) ===
# Copied VERBATIM from the base notebook. NOTE: `states_and_top` and `arith_fullanswer_correct`
# below close over a global `tokenizer`; CELL 6 aliases `tokenizer = tokenizer_r` so they remain
# correct if called, but the CROSS-FAMILY code in this notebook uses the explicitly
# tokenizer-parameterized replacements defined in CELL X0 instead.
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2

# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out

import re as _re
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None

# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok

print("helpers defined")

helpers defined


In [7]:
# === CELL X0: CROSS-FAMILY (dual-tokenizer) helpers — run right after CELL 5 ===
# Everything here exists because donor and recipient DO NOT share a tokenizer.
#
# DESIGN DECISION 1 — one problem, two encodings.
#   Each problem dict carries ids_d/tok_d (donor tokenization) and ids_r/tok_r (recipient
#   tokenization) of the SAME text. `ids_*` ends at that model's own LAST PROMPT TOKEN, i.e. the
#   position whose logits predict the first answer token. On Gemma that is the standalone leading
#   space; on Qwen/Llama the space fuses with the first digit so it is the "=" token. Different
#   surface positions, identical operative role. Only this ONE position is grafted, so the two
#   token sequences never need to align.
#
# DESIGN DECISION 2 — scoring is tokenizer-agnostic.
#   No token ID is compared across models anywhere. gen_score_arith greedily generates from the
#   recipient and scores the DECODED text: full-answer correctness and leading-digit correctness.
#   The old "first-token conferral" metric is NOT valid cross-family and is not reported as a
#   headline. `tok_r` is still used, but only as the CE target when training the map — that is a
#   purely within-recipient quantity in the recipient's own vocabulary, which is legitimate.
import torch, torch.nn.functional as F, numpy as np, random, json

def fd(x):
    """First (leading) digit of an integer answer, 0-9. Defined here because the notebooks this
    is assembled from define it only in cells we do not include."""
    return int(str(abs(int(round(x))))[0]) if x is not None else 0

def probe_first_digit(Xtr, ytr, Xte, yte, steps=300, nclass=10, lr=1e-2):
    """Linear probe on a hidden state -> class label. Returns PREDICTIONS on Xte."""
    Pw = torch.zeros(Xtr.shape[1], nclass, requires_grad=True)
    opt = torch.optim.Adam([Pw], lr=lr); mu = Xtr.mean(0); A, Bx = Xtr-mu, Xte-mu
    for _ in range(steps):
        opt.zero_grad(); F.cross_entropy(A @ Pw, ytr).backward(); opt.step()
    return (Bx @ Pw.detach()).argmax(1)

def probe_split_acc(X, y, nclass=10):
    """Half/half split probe accuracy as a Wilson-CI string. X, y are CPU tensors."""
    if len(y) < 40: return "n/a (n<40)"
    h = len(y)//2
    pred = probe_first_digit(X[:h].float(), y[:h], X[h:].float(), y[h:], nclass=nclass)
    return fmt(wilson_bools((pred == y[h:]).tolist()))

# ---- tokenizer-parameterized state collection (replaces CELL 5's states_and_top) ----
@torch.inference_mode()
def states_and_top_tok(model, tok, layers, prob_ids, batch=None):
    """Left-pad with THIS model's own pad id, forward once, take the last-position residual at
    each requested layer. Returns ({layer: [N,d]}, top_token_ids). The top ids are a diagnostic
    only; they are never compared across models."""
    batch = batch or ARITH_BATCH
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tok.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- arithmetic data with BOTH encodings ----
def gen_arith_dual(n, rng, exclude=None):
    """Generate a*b+c problems that BOTH tokenizers can encode, so donor and recipient see an
    identical problem set. Drops a problem if either tokenizer is not prefix-consistent."""
    exclude = exclude or set(); out, seen = [], set(); tries = 0; dropped = 0
    while len(out) < n and tries < n*200:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids_d, tid_d = _aencode(tokenizer_d, expr, ans)
        ids_r, tid_r = _aencode(tokenizer_r, expr, ans)
        if tid_d is None or tid_r is None:
            dropped += 1; continue
        out.append(dict(expr=expr, ans=ans, ids_d=ids_d, tok_d=tid_d, ids_r=ids_r, tok_r=tid_r))
    if dropped: print(f"  gen_arith_dual: dropped {dropped} problems (tokenizer prefix-inconsistent)")
    return out

# ---- THE scoring function: decoded-answer metrics only ----
@torch.inference_mode()
def gen_score_arith(model, tok, layer, probs, ids_key, vecs=None, batch=None, max_new=None):
    """Greedy-generate from `model` using its OWN encoding (probs[i][ids_key]) and score the
    DECODED text. If vecs is given, vecs[i] is grafted at the last prompt position during prefill.
    Returns {"full": [bool], "lead": [bool]} -- (a) whole-answer string correctness,
    (b) leading-digit correctness. Tokenizer-agnostic by construction."""
    batch = batch or ARITH_BATCH; max_new = max_new or MAX_NEW_ARITH
    full, lead, handle = [], [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p[ids_key] for p in chunk], tok.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=max_new,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            txt = tok.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                full.append(pred is not None and abs(pred - p["ans"]) < 0.5)
                lead.append(pred is not None and fd(pred) == fd(p["ans"]))
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return {"full": full, "lead": lead}

def score_pair(sc):
    """Format a {'full':..,'lead':..} score dict as two Wilson-CI strings."""
    return {"full": fmt(wilson_bools(sc["full"])), "lead": fmt(wilson_bools(sc["lead"]))}

print("cross-family helpers defined (fd, probe_first_digit, probe_split_acc, "
      "states_and_top_tok, gen_arith_dual, gen_score_arith, score_pair)")

cross-family helpers defined (fd, probe_first_digit, probe_split_acc, states_and_top_tok, gen_arith_dual, gen_score_arith, score_pair)


In [8]:
# === CELL G0: the geometry metrics — the point of this notebook ===
# cosine alone cannot tell "on the manifold" from "parallel but 6x too long". These three
# together can. z_dist is the one to report: native states score ~1.0 by construction, so a
# value of k means the written vector sits k times further from the native mean, per dimension,
# than a typical native state does.
def geometry(V, N, label=""):
    """V = written vectors [n,d] (cpu float), N = recipient native states [n,d] (cpu float)."""
    V, N = V.float(), N.float()
    d = V.shape[1]
    cos   = F.cosine_similarity(V, N, dim=1).numpy()
    nv    = V.norm(dim=1).numpy()
    nn_   = N.norm(dim=1).numpy()
    ratio = nv / (nn_ + 1e-9)
    mu, sd = N.mean(0), N.std(0) + 1e-6
    zV = (((V - mu) / sd).norm(dim=1) / math.sqrt(d)).numpy()   # graft, in native SD units
    zN = (((N - mu) / sd).norm(dim=1) / math.sqrt(d)).numpy()   # native baseline (~1.0)
    def q(a): return [round(float(np.percentile(a, p)), 4) for p in (10, 50, 90)]
    return {
        "cos_mean": round(float(cos.mean()), 4),           "cos_p10_p50_p90": q(cos),
        "norm_written_mean": round(float(nv.mean()), 2),
        "norm_native_mean":  round(float(nn_.mean()), 2),
        "norm_ratio_mean":   round(float(ratio.mean()), 3), "norm_ratio_p10_p50_p90": q(ratio),
        "z_dist_written":    round(float(zV.mean()), 3),    "z_dist_written_p10_p50_p90": q(zV),
        "z_dist_native":     round(float(zN.mean()), 3),
        "z_excess_over_native": round(float(zV.mean() / (zN.mean() + 1e-9)), 3),
        "n": int(V.shape[0]),
    }

def fmt_geom(g):
    return (f"cos {g['cos_mean']:.3f} | norm {g['norm_written_mean']:.0f} vs native "
            f"{g['norm_native_mean']:.0f} (x{g['norm_ratio_mean']:.2f}) | z {g['z_dist_written']:.2f} "
            f"vs {g['z_dist_native']:.2f} (x{g['z_excess_over_native']:.2f})")
print("geometry metrics defined")

geometry metrics defined


In [9]:
# === CELL G1: the arm loop — load donor, train, measure geometry, free, repeat ===
# The recipient loads ONCE and stays resident; donors load and free one at a time, so peak VRAM
# is recipient(5GB) + largest donor(18.5GB) ~= 24GB. Everything downstream of the map training
# reuses the exact recipe from the completed cross-family runs, so the cosines are comparable.
import gc

tokenizer_r = AutoTokenizer.from_pretrained(MODEL_R)
tokenizer_r.padding_side = "left"
if tokenizer_r.pad_token is None: tokenizer_r.pad_token = tokenizer_r.eos_token
tokenizer = tokenizer_r                      # back-compat for helpers closing over `tokenizer`
model_r = AutoModelForCausalLM.from_pretrained(
    MODEL_R, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
model_r.requires_grad_(False)
D_RECIP = model_r.config.hidden_size
print(f"recipient {MODEL_R}: {model_r.config.num_hidden_layers} layers, d={D_RECIP}")

ARM_BY_NAME = {n: (m, l, c) for n, m, l, c in ARMS}

for ARM in RUN_ARMS:
    MODEL_D, L_D, PUB_COS = ARM_BY_NAME[ARM]
    print("\n" + "=" * 88); print(f"ARM {ARM}: donor {MODEL_D} L{L_D} -> {MODEL_R} L{L_R}"); print("=" * 88)

    tokenizer_d = AutoTokenizer.from_pretrained(MODEL_D)
    tokenizer_d.padding_side = "left"
    if tokenizer_d.pad_token is None: tokenizer_d.pad_token = tokenizer_d.eos_token
    model_d = AutoModelForCausalLM.from_pretrained(
        MODEL_D, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
    model_d.requires_grad_(False)
    assert 0 <= L_D < model_d.config.num_hidden_layers, f"L_D={L_D} out of range"
    with torch.inference_mode():
        _hl = model_d(tokenizer_d("The capital of France is", return_tensors="pt")
                      .to(DEVICE).input_ids).logits[0, -1, :]
    assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), "donor NaN/Inf logits"
    del _hl
    print(f"  donor loaded: {model_d.config.num_hidden_layers} layers, d={model_d.config.hidden_size}")

    # ---- problems, encoded separately for each model (cross-family safe) ----
    rng_t, rng_e = random.Random(0), random.Random(1)
    train = gen_arith_dual(N_ARITH_TRAIN, rng_t)
    evalp = gen_arith_dual(N_ARITH_EVAL, rng_e, {p["expr"] for p in train})
    Xd_t, _        = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in train])
    Xd_e, _        = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in evalp])
    Xr_t, _        = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in train])
    Xr_e, r_top    = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in evalp])
    Xd_t, Xd_e, Xr_t, Xr_e = Xd_t[L_D], Xd_e[L_D], Xr_t[L_R], Xr_e[L_R]

    donor_ok  = gen_score_arith(model_d, tokenizer_d, L_D, evalp, "ids_d", vecs=None)["full"]
    keep      = [i for i, ok in enumerate(donor_ok) if ok]
    evalp     = [evalp[i] for i in keep]; Xd_e = Xd_e[keep]; Xr_e = Xr_e[keep]
    native_ok = gen_score_arith(model_r, tokenizer_r, L_R, evalp, "ids_r", vecs=None)["full"]
    unsolv    = [i for i, ok in enumerate(native_ok) if not ok]
    print(f"  donor-solved {len(evalp)} | recipient-unsolvable {len(unsolv)}")

    mu_d, mu_r, Wr = fit_ridge(Xd_t, Xr_t)
    mu_dd = mu_d.to(DEVICE)
    def map_recon(x): return (x.to(DEVICE) - mu_dd) @ Wr.to(DEVICE) + mu_r.to(DEVICE)

    # ---- train task maps, logging geometry EVERY epoch (the undertraining test) ----
    Xd_sub, Xr_sub = Xd_e[unsolv], Xr_e[unsolv]
    def _geom_now(W, b):
        with torch.no_grad():
            V = ((Xd_sub.to(DEVICE) - mu_dd) @ W + b).cpu()
        return geometry(V, Xr_sub)
    task_maps, traj = [], []
    n_ep = TASK_EPOCHS + (EXTENDED_EPOCHS if ARM in EXTEND_ARMS else 0)
    for si, seed in enumerate(GEOM_SEEDS):
        torch.manual_seed(seed); random.seed(seed)
        W = Wr.clone().to(DEVICE).requires_grad_(True)
        b = mu_r.clone().to(DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([W, b], lr=1e-3)
        hd = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
        idx = list(range(len(train)))
        try:
            for ep in range(n_ep):
                random.Random(seed * 100 + ep).shuffle(idx)
                last = 0.0
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s + ARITH_BATCH]
                    _graft["vec"] = (Xd_t[sub].to(DEVICE) - mu_dd) @ W + b
                    ids, m = left_pad([train[k]["ids_r"] for k in sub], tokenizer_r.pad_token_id)
                    lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([train[k]["tok_r"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
                    last = float(loss.item())
                if si == 0:
                    g = _geom_now(W.detach(), b.detach())
                    traj.append({"epoch": ep + 1, "ce": round(last, 4), "cos": g["cos_mean"],
                                 "norm_ratio": g["norm_ratio_mean"], "z": g["z_dist_written"],
                                 "phase": "standard" if ep < TASK_EPOCHS else "extended"})
                    print(f"    seed{seed} ep{ep+1:2d}: CE {last:.4f} cos {g['cos_mean']:.4f} "
                          f"ratio {g['norm_ratio_mean']:.2f} z {g['z_dist_written']:.2f}", flush=True)
        finally:
            hd.remove(); _graft["vec"] = None
        task_maps.append((W.detach(), b.detach()))

    def map_task(i):
        W, b = task_maps[i]
        return lambda x: (x.to(DEVICE) - mu_dd) @ W + b

    # ---- geometry on the unsolvable bin ----
    with torch.no_grad():
        V_recon = map_recon(Xd_sub).cpu()
        V_task  = map_task(0)(Xd_sub).cpu()
    g_recon, g_task = geometry(V_recon, Xr_sub), geometry(V_task, Xr_sub)
    per_seed_cos = []
    for i in range(len(task_maps)):
        with torch.no_grad():
            per_seed_cos.append(round(float(F.cosine_similarity(
                map_task(i)(Xd_sub).cpu().float(), Xr_sub.float(), dim=1).mean()), 4))
    print(f"  recon: {fmt_geom(g_recon)}")
    print(f"  task : {fmt_geom(g_task)}")
    print(f"  per-seed task cosine: {per_seed_cos}  (published {PUB_COS})")

    # ---- conferral, to confirm the map actually works ----
    def _confer(mapfn, idxs):
        vecs = list(mapfn(Xd_e[idxs]).detach().float().cpu())
        return gen_score_arith(model_r, tokenizer_r, L_R, [evalp[j] for j in idxs], "ids_r",
                               vecs=vecs)["lead"]
    conf = {"recon": {"lead": round(float(np.mean(_confer(map_recon, unsolv))), 4)},
            "task":  {"lead": round(float(np.mean(_confer(map_task(0), unsolv))), 4)}}
    print(f"  conferral (lead): recon {conf['recon']['lead']}  task {conf['task']['lead']}")

    verdict = None
    if ARM in EXTEND_ARMS and traj:
        c_std = [r["cos"] for r in traj if r["phase"] == "standard"][-1]
        c_ext = [r["cos"] for r in traj if r["phase"] == "extended"]
        if c_ext:
            drop = c_std - c_ext[-1]
            verdict = (f"cosine {c_std:.4f} -> {c_ext[-1]:.4f} over {len(c_ext)} extra epochs "
                       f"(drop {drop:.4f}). " +
                       ("STILL FALLING: the map was undertrained, the anomaly is a training artefact."
                        if drop > 0.05 else
                        "CONVERGED: the cosine is a real optimum, not undertraining."))
            print(f"  {verdict}")

    RESULTS[ARM] = {"donor": MODEL_D, "L_D": L_D, "L_R": L_R,
                    "n_donor_solved": len(evalp), "n_unsolvable": len(unsolv),
                    "published_taskmap_cos": PUB_COS,
                    "geometry": {"recon": g_recon, "task": g_task},
                    "per_seed_task_cos": per_seed_cos,
                    "conferral": conf, "trajectory": traj, "extended_verdict": verdict,
                    "epochs_run": n_ep}

    del model_d, tokenizer_d, Xd_t, Xd_e, Xr_t, Xr_e, task_maps, Wr, mu_d, mu_r, mu_dd
    gc.collect(); torch.cuda.empty_cache()
    print(f"  donor freed | VRAM now {torch.cuda.memory_allocated()/2**30:.1f} GB")
print("\nall arms done")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

recipient google/gemma-2-2b: 26 layers, d=2304

ARM gemma9b: donor google/gemma-2-9b L34 -> google/gemma-2-2b L20


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

  donor loaded: 42 layers, d=3584
  donor-solved 699 | recipient-unsolvable 604
    seed0 ep 1: CE 0.3267 cos 0.3175 ratio 3.37 z 6.52
    seed0 ep 2: CE 0.1915 cos 0.2937 ratio 3.73 z 7.23
    seed0 ep 3: CE 0.1804 cos 0.2669 ratio 4.12 z 7.98
    seed0 ep 4: CE 0.0923 cos 0.2141 ratio 5.11 z 9.74
    seed0 ep 5: CE 0.0369 cos 0.1975 ratio 5.58 z 10.60
    seed0 ep 6: CE 0.0269 cos 0.1939 ratio 5.72 z 10.87
    seed0 ep 7: CE 0.0433 cos 0.1709 ratio 6.51 z 12.35
    seed0 ep 8: CE 0.2546 cos 0.1694 ratio 6.54 z 12.36
    seed0 ep 9: CE 0.0733 cos 0.1699 ratio 6.44 z 12.16
    seed0 ep10: CE 0.0709 cos 0.1694 ratio 6.44 z 12.17
    seed0 ep11: CE 0.1038 cos 0.1573 ratio 7.03 z 13.34
    seed0 ep12: CE 0.0028 cos 0.1577 ratio 6.95 z 13.13
  recon: cos 0.978 | norm 291 vs native 296 (x0.98) | z 0.93 vs 0.98 (x0.95)
  task : cos 0.158 | norm 2055 vs native 296 (x6.95) | z 13.13 vs 0.98 (x13.41)
  per-seed task cosine: [0.1577, 0.1363]  (published 0.197)
  conferral (lead): recon 0.5844  t

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

  donor loaded: 32 layers, d=4096


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  donor-solved 297 | recipient-unsolvable 236
    seed0 ep 1: CE 0.6216 cos 0.8987 ratio 0.93 z 0.82
    seed0 ep 2: CE 0.1005 cos 0.8763 ratio 0.96 z 0.94
    seed0 ep 3: CE 0.0273 cos 0.8583 ratio 0.98 z 1.04
    seed0 ep 4: CE 0.1159 cos 0.8429 ratio 1.00 z 1.12
    seed0 ep 5: CE 0.0092 cos 0.8294 ratio 1.02 z 1.20
    seed0 ep 6: CE 0.0048 cos 0.8189 ratio 1.04 z 1.25
    seed0 ep 7: CE 0.4360 cos 0.8069 ratio 1.06 z 1.31
    seed0 ep 8: CE 0.0139 cos 0.7983 ratio 1.07 z 1.35
    seed0 ep 9: CE 0.0516 cos 0.7903 ratio 1.09 z 1.40
    seed0 ep10: CE 0.0015 cos 0.7852 ratio 1.09 z 1.42
    seed0 ep11: CE 1.2051 cos 0.7792 ratio 1.10 z 1.45
    seed0 ep12: CE 0.0025 cos 0.7735 ratio 1.12 z 1.48
  recon: cos 0.932 | norm 264 vs native 299 (x0.89) | z 0.60 vs 0.98 (x0.62)
  task : cos 0.773 | norm 333 vs native 299 (x1.12) | z 1.48 vs 0.98 (x1.51)
  per-seed task cosine: [0.7735, 0.7776]  (published 0.808)
  conferral (lead): recon 0.572  task 0.8814
  cosine 0.8189 -> 0.7735 over 6 ex

In [10]:
# === CELL SAVE: comparison table + JSON ===
print("=" * 100)
print("OFF-DISTRIBUTION GEOMETRY — task map vs recon map, per arm")
print("=" * 100)
hdr = f"{'arm':10s} {'map':6s} {'cos':>7s} {'|f(h)|':>9s} {'|native|':>9s} {'ratio':>7s} {'z':>7s} {'z/native':>9s} {'confer':>8s}"
print(hdr); print("-" * len(hdr))
for name in RUN_ARMS:
    a = RESULTS.get(name)
    if not a: continue
    for mp in ("recon", "task"):
        g = a.get("geometry", {}).get(mp)
        if not g: continue
        cf = a.get("conferral", {}).get(mp, {}).get("lead", "")
        print(f"{name:10s} {mp:6s} {g['cos_mean']:7.3f} {g['norm_written_mean']:9.0f} "
              f"{g['norm_native_mean']:9.0f} {g['norm_ratio_mean']:7.2f} {g['z_dist_written']:7.2f} "
              f"{g['z_excess_over_native']:9.2f} {str(cf):>8s}")
print()
print("READ: if llama8b's task map has a norm ratio and z-excess comparable to gemma9b's, then it")
print("      IS off-distribution -- in scale rather than direction -- and the manifold claim holds")
print("      in a stronger restated form. If its ratio and z are ~1, it is genuinely on-manifold")
print("      and is a real counterexample that bounds Sec 6.3.")
print()
for name in RUN_ARMS:
    tr = RESULTS.get(name, {}).get("trajectory")
    if not tr: continue
    print(f"--- {name}: training trajectory (seed {GEOM_SEEDS[0]}) ---")
    for r in tr:
        print(f"    epoch {r['epoch']:2d}  CE {r['ce']:.4f}  cos {r['cos']:.4f}  "
              f"norm_ratio {r['norm_ratio']:.2f}  z {r['z']:.2f}")
    ext = RESULTS[name].get("extended_verdict")
    if ext: print(f"    VERDICT: {ext}")
    print()
fname = "manifold_geometry_check.json"
with open(fname, "w") as f: json.dump(RESULTS, f, indent=2)
print("saved", fname, "->  DOWNLOAD before terminating the pod")

OFF-DISTRIBUTION GEOMETRY — task map vs recon map, per arm
arm        map        cos    |f(h)|  |native|   ratio       z  z/native   confer
--------------------------------------------------------------------------------
gemma9b    recon    0.978       291       296    0.98    0.93      0.95   0.5844
gemma9b    task     0.158      2055       296    6.95   13.13     13.41   0.9354
qwen7b     recon    0.973       287       295    0.97    0.89      0.91   0.5444
qwen7b     task     0.254      1271       295    4.31    8.30      8.49   0.8155
llama8b    recon    0.932       264       299    0.89    0.60      0.62    0.572
llama8b    task     0.773       333       299    1.12    1.48      1.51   0.8814

READ: if llama8b's task map has a norm ratio and z-excess comparable to gemma9b's, then it
      IS off-distribution -- in scale rather than direction -- and the manifold claim holds
      in a stronger restated form. If its ratio and z are ~1, it is genuinely on-manifold
      and is a real